# Onset-HFO 7 — Mark a few hundred windows, and find out what the reference is worth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/onset-hfo/blob/master/onset-hfo/notebooks/07_annotation.ipynb)

Every accuracy number in this project is agreement with *another algorithm* —
the archive's own Morphology detector — or accuracy against a simulator that
can only measure what it simulates. "Precision 0.97 on synthetic ripples" is a
statement about the generator.

**This notebook turns that into a statement about recordings**, by having two
people mark real windows. It is the one measurement here that needs humans,
and it takes two reviewers an afternoon each.

**The three things that make such a benchmark worth doing** are all in the
sampling rather than the marking, and `onset_hfo.review` enforces them:

1. **Windows the detector never proposed are in the sample.** Review only the
   detector's own hits and you can measure precision and *never* recall — and
   the reference becomes a function of the thing you are scoring.
2. **The reviewer cannot see the detector's verdict.** The manifest carries
   four columns: an opaque id, a channel, and two times. The stratum and the
   detector's opinion go in a separate key file, named so that handing over
   the wrong one is a visible mistake.
3. **Agreement is chance-corrected.** Most windows are not ripples, so two
   reviewers who say "no" to everything agree ~90% of the time. Cohen's kappa,
   with a bootstrap interval, and the prevalence beside it.

**And the ceiling is the finding.** Two reviewers who agree at kappa 0.6 have
defined a reference no detector can be scored against more finely than that.
The notebook prints that sentence before it prints any score.

**Runtime.** Minutes of compute; the marking is however long you take.
Progress is appended to a CSV after every window, so you can close the tab.

In [ ]:
# Colab setup. On your own machine, skip this cell and run
#   pip install -e ".[dev]"  from the onset-hfo directory instead.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/berdakh/onset-hfo.git"
BRANCH = "master"   # change to the default branch once this work is merged

if IN_COLAB and not os.path.exists("onset-hfo"):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1", REPO], check=True)
if IN_COLAB:
    os.chdir("/content/onset-hfo/onset-hfo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

## 1. Something to review

Any saved analysis will do. The shipped example is a real 60-second recording
from OpenNeuro `ds003029` and needs no download.

In [ ]:
from onset_hfo.store import ResultStore
from onset_hfo.detectors.base import Event

store = ResultStore("data/example_analysis")
events = store.events
by_detector = {}
for name in ("rms", "line_length"):
    rows = events[events.detector == name]
    by_detector[name] = [
        Event(channel=r.channel, start=r.start, stop=r.stop, detector=name,
              band=(r.band_low, r.band_high), accepted=bool(r.accepted))
        for r in rows.itertuples()]

channels = sorted(events.channel.unique())
duration = float(store.provenance.get("slice_stop_s", 60)
                 - store.provenance.get("slice_start_s", 0))
print(f"{sum(len(v) for v in by_detector.values())} candidates on "
      f"{len(channels)} channels over {duration:.0f} s")

## 2. Draw a blinded, stratified sample

Equal numbers from three strata **by design, not by prevalence**: background
outnumbers candidates enormously in a real recording, and sampling it
proportionally would spend the whole afternoon on empty windows. The imbalance
is corrected when scoring, which is why the stratum is recorded.

Set `N_PER_STRATUM` to what your reviewers will actually finish. 70 each means
210 windows, which is roughly an afternoon.

In [ ]:
from pathlib import Path
from onset_hfo.review import sample_windows, write_manifest

N_PER_STRATUM = 70
OUT = Path("artifacts/review")

windows = sample_windows(by_detector, duration_s=duration, channels=channels,
                         n_per_stratum=N_PER_STRATUM, seed=0)
manifest_path = write_manifest(windows, OUT / "manifest.csv")

import pandas as pd
print(pd.Series([w.stratum for w in windows]).value_counts().to_string())
print()
print("give the reviewer:", manifest_path)
print("do NOT give them:", manifest_path.with_suffix(".key.csv"))

## 3. Mark them

One window at a time: the wideband trace, the band-passed trace, and the
event's own spectrum. The question is whether it is an oscillation with a
spectral bump of its own, or a sharp transient the filter turned into one.

**`unsure` is a real answer.** Forcing a binary choice on an ambiguous window
manufactures agreement that is not there, and the analysis drops those
windows explicitly and counts them.

Set `REVIEWER` to your own initials. Re-running the cell resumes where you
stopped.

In [ ]:
REVIEWER = "A"   # <- change per reviewer

import ipywidgets as widgets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

from onset_hfo.preprocess import prepare
from onset_hfo.review import AnnotationLog, read_manifest
from onset_hfo.viz import plot_event

manifest = read_manifest(manifest_path)
log = AnnotationLog(OUT / "annotations.csv", reviewer=REVIEWER)
todo = log.remaining(manifest)
print(f"{log.done} done, {len(todo)} to go")

# The signal behind the windows. `reload_spec` on a saved analysis says which
# slice to fetch; here we use the analysis the store was built from.
prep = prepare(store.recording(), verbose=False) if hasattr(store, "recording") else None

If `prep` is `None` above, the saved analysis does not carry its signal and
the windows cannot be drawn — run `python -m onset_hfo.cli run` first, or
point `ResultStore` at an analysis you produced locally.

The loop below draws one window and waits for a click.

In [ ]:
out = widgets.Output()
state = {"i": 0}

def show():
    with out:
        clear_output(wait=True)
        if state["i"] >= len(todo):
            print(f"Done — {log.done} windows marked by {REVIEWER}.")
            return
        row = manifest[manifest.window_id == todo[state["i"]]].iloc[0]
        mid = 0.5 * (row.t_start + row.t_stop)
        window = Event(channel=row.channel, start=mid - 0.015, stop=mid + 0.015,
                       detector="", band=(80.0, 250.0))
        print(f"{row.window_id}   ({state['i'] + 1} of {len(todo)})")
        plot_event(prep, window, context_s=row.t_stop - row.t_start)
        plt.show()

def mark(label):
    def handler(_):
        log.record(todo[state["i"]], label)
        state["i"] += 1
        show()
    return handler

buttons = []
for label, style in (("hfo", "success"), ("not_hfo", "danger"), ("unsure", "warning")):
    b = widgets.Button(description=label, button_style=style)
    b.on_click(mark(label))
    buttons.append(b)

display(widgets.HBox(buttons), out)
show()

## 4. What the reference is worth

Run this once **both** reviewers have finished. The ceiling sentence comes
first on purpose: every score after it has to be read against that number.

In [ ]:
from onset_hfo.review import agreement, ceiling_note

annotations = pd.read_csv(OUT / "annotations.csv")
stats = agreement(annotations)
print(ceiling_note(stats))
print()
for k, v in stats.items():
    print(f"  {k}: {v}")

## 5. Score the detector against it

Per stratum, never pooled: the strata were sampled in equal numbers rather
than in proportion, so a pooled figure would weight a background window as
heavily as a candidate and describe a recording that does not exist.

Both consensus rules are printed because they bracket the answer. Quoting
whichever is kinder is how a detector's precision gets published.

In [ ]:
from onset_hfo.review import score_against_annotations, save_study

key = pd.read_csv(manifest_path.with_suffix(".key.csv"))
for rule in ("both", "either"):
    scores = score_against_annotations(by_detector["rms"], manifest, annotations,
                                       key, consensus=rule, detector="rms")
    print(f"consensus = {rule}")
    for stratum, s in scores["strata"].items():
        print(f"  {stratum:11s} n={s['n']:3d}  tp={s['true_positive']:3d} "
              f"fp={s['false_positive']:3d} fn={s['false_negative']:3d} "
              f"tn={s['true_negative']:3d}  reviewers called "
              f"{s['reviewer_hfo_rate']:.0%} of them HFOs")
    print()

save_study(OUT, stats, scores)
print("written to", OUT)

## What to do with it

**Publish the annotations.** The marks are the contribution — a scored
detector is reproducible from them, and nobody else has to spend the
afternoon. Include the manifest, the annotations and the key.

**Publish the kappa next to every score.** A detector reported at precision
0.8 against a reference whose reviewers agreed at kappa 0.4 has been measured
against something largely made of disagreement, and the number will not
survive a second pair of reviewers.

**Do not pool the strata.** The sample is not a recording.